# Notebook 08: Transactions & Pipelines — Atomic Operations

Sometimes you need to:
- Run **multiple commands as a single atomic unit** (transactions)
- **Batch many commands** for performance (pipelines)

Redis provides both, and understanding the difference is crucial.

In [ ]:
import redis
import time

r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
r.flushdb()
print("Connected and ready!")

---
## Part 1: Pipelines — Speed Through Batching

### The Problem

Each Redis command = 1 network **round trip** (send command → wait → receive response).

```
Without pipeline (3 round trips):
Client → SET a 1 → Server → OK → Client
Client → SET b 2 → Server → OK → Client
Client → SET c 3 → Server → OK → Client

With pipeline (1 round trip):
Client → [SET a 1, SET b 2, SET c 3] → Server → [OK, OK, OK] → Client
```

Pipeline is like sending a batch of letters instead of one at a time.

In [ ]:
# Basic Pipeline
pipe = r.pipeline(transaction=False)  # transaction=False for pure batching

# Queue up commands (nothing is sent yet!)
pipe.set('a', 1)
pipe.set('b', 2)
pipe.set('c', 3)
pipe.get('a')
pipe.get('b')
pipe.get('c')

# NOW send everything and get all results
results = pipe.execute()
print(f"Results: {results}")
# [True, True, True, '1', '2', '3']
# First 3: SET results, Last 3: GET results

### Pipeline Performance Benchmark

Let's see the dramatic speedup!

In [ ]:
r.flushdb()
NUM_OPS = 5000

# Without pipeline — individual commands
start = time.time()
for i in range(NUM_OPS):
    r.set(f'key:{i}', f'value_{i}')
time_without = time.time() - start

r.flushdb()

# With pipeline — batched commands
start = time.time()
pipe = r.pipeline(transaction=False)
for i in range(NUM_OPS):
    pipe.set(f'key:{i}', f'value_{i}')
pipe.execute()
time_with = time.time() - start

speedup = time_without / time_with
print(f"Without pipeline: {time_without:.3f}s")
print(f"With pipeline:    {time_with:.3f}s")
print(f"Speedup:          {speedup:.1f}x faster!")

### Pipeline as Context Manager

In [ ]:
r.flushdb()

# Cleaner syntax with 'with' statement
with r.pipeline(transaction=False) as pipe:
    pipe.set('user:1:name', 'Sujit')
    pipe.set('user:1:email', 'sujit@test.com')
    pipe.set('user:1:age', 25)
    pipe.get('user:1:name')
    results = pipe.execute()

print(f"Results: {results}")
print(f"Name: {results[3]}")  # 4th result is the GET

### Pipeline Limitation

You **cannot** use the result of one command inside the same pipeline. The results are only available after `execute()`.

In [ ]:
# This WON'T work as you might expect:
# pipe.get('key')  → This returns a Pipeline object, not the value!
# pipe.set('other', <result>)  → Can't use the result yet

# For conditional logic, you need Transactions with WATCH (below)

---
## Part 2: Transactions (MULTI/EXEC) — Atomicity

A transaction groups commands so they run:
- **Sequentially** — one after another, in order
- **Atomically** — no other client's commands can slip in between

It's like putting commands in an envelope: Redis opens the envelope and runs everything inside without interruption.

In [ ]:
r.flushdb()

# Transaction = Pipeline with transaction=True (default)
pipe = r.pipeline()  # transaction=True by default!

# MULTI — Start transaction
pipe.multi()

# Queue commands (these are QUEUED, not executed yet)
pipe.set('account:A', 1000)
pipe.set('account:B', 500)
pipe.decrby('account:A', 200)  # Transfer $200 from A
pipe.incrby('account:B', 200)  # to B

# EXEC — Execute all commands atomically
results = pipe.execute()
print(f"Transaction results: {results}")

print(f"Account A: ${r.get('account:A')}")
print(f"Account B: ${r.get('account:B')}")
print("\nBoth accounts updated atomically — no partial state visible!")

### What Transactions Guarantee

- **Isolation:** No other client sees intermediate state
- **Atomicity:** All commands run, or none run (if EXEC is called)

### What Transactions Do NOT Guarantee

- **No ROLLBACK:** If one command fails, the others still execute!
- This is by design: errors in a transaction are programmer mistakes, not runtime issues

In [ ]:
# Demonstrating: errors don't roll back other commands
r.set('mykey', 'hello')  # This is a string

pipe = r.pipeline()
pipe.multi()
pipe.set('a', 1)         # This will succeed
pipe.incr('mykey')       # This will FAIL (can't INCR a non-numeric string)
pipe.set('b', 2)         # This will still succeed!

try:
    results = pipe.execute()
except redis.ResponseError as e:
    # execute() raises if any command fails, but still runs all commands
    print(f"Error: {e}")

# Let's check with raise_on_error=False
r.set('mykey', 'hello')
pipe = r.pipeline()
pipe.multi()
pipe.set('a', 1)
pipe.incr('mykey')  # Will fail
pipe.set('b', 2)
results = pipe.execute(raise_on_error=False)

print(f"\nResults (with errors): {results}")
print(f"a = {r.get('a')}")  # '1' — succeeded
print(f"b = {r.get('b')}")  # '2' — succeeded despite error in middle

---
## WATCH — Optimistic Locking

This is the most important concept! WATCH enables **safe read-then-write** operations.

**The Problem:** What if you want to:
1. Read account balance
2. Check if sufficient funds
3. Deduct amount

Between steps 1 and 3, another client might change the balance!

**WATCH** monitors a key. If anyone changes it before your EXEC, the transaction is **automatically cancelled**.

In [ ]:
r.flushdb()
r.set('account:A', 1000)
r.set('account:B', 500)

def transfer_money(from_acc, to_acc, amount):
    """Safely transfer money using WATCH for optimistic locking."""
    with r.pipeline() as pipe:
        while True:  # Retry loop
            try:
                # WATCH the source account
                pipe.watch(f'account:{from_acc}')
                
                # READ the current balance (outside transaction)
                balance = int(pipe.get(f'account:{from_acc}'))
                
                if balance < amount:
                    pipe.unwatch()  # Cancel the watch
                    return False, f"Insufficient funds (have {balance}, need {amount})"
                
                # Start the transaction
                pipe.multi()
                
                # WRITE (these are queued)
                pipe.decrby(f'account:{from_acc}', amount)
                pipe.incrby(f'account:{to_acc}', amount)
                
                # EXEC — if watched key changed, raises WatchError
                pipe.execute()
                return True, f"Transferred ${amount} from {from_acc} to {to_acc}"
                
            except redis.WatchError:
                # Someone else modified the account! Retry.
                print("  [WatchError] Account was modified by another client, retrying...")
                continue

# Transfer $300 from A to B
success, msg = transfer_money('A', 'B', 300)
print(f"Transfer: {msg}")
print(f"Account A: ${r.get('account:A')}")
print(f"Account B: ${r.get('account:B')}")

# Try to transfer more than available
success, msg = transfer_money('A', 'B', 9999)
print(f"\nOverdraft: {msg}")

### WATCH Pattern Summary

```
1. WATCH key(s)         ← Monitor for changes
2. Read current values   ← Make decisions based on current state
3. MULTI                 ← Start transaction
4. Queue write commands   ← Based on what you read
5. EXEC                  ← If watched keys changed → WatchError → retry
```

This is called **optimistic locking** because you optimistically assume no one will interfere, and just retry if they do.

---
## Real-World: Inventory Purchase

In [ ]:
r.flushdb()

# Set up inventory
r.hset('product:laptop', mapping={'name': 'MacBook Pro', 'price': '1999', 'stock': '5'})

def purchase(product_id, user_id, quantity=1):
    """Safely purchase a product with inventory check."""
    key = f'product:{product_id}'
    
    with r.pipeline() as pipe:
        while True:
            try:
                pipe.watch(key)
                
                # Read current stock
                stock = int(pipe.hget(key, 'stock'))
                price = float(pipe.hget(key, 'price'))
                
                if stock < quantity:
                    pipe.unwatch()
                    return False, f"Only {stock} in stock"
                
                # Execute purchase atomically
                pipe.multi()
                pipe.hincrby(key, 'stock', -quantity)
                pipe.rpush(f'orders:{user_id}', 
                          f'{product_id}:qty={quantity}:total=${price * quantity}')
                pipe.execute()
                
                return True, f"Purchased {quantity}x {product_id} for ${price * quantity}"
                
            except redis.WatchError:
                continue

# Simulate purchases
for i in range(7):  # Try to buy 7, but only 5 in stock
    success, msg = purchase('laptop', f'user_{i}')
    status = 'OK' if success else 'FAIL'
    print(f"  [{status}] User {i}: {msg}")

print(f"\nRemaining stock: {r.hget('product:laptop', 'stock')}")

---
## Real-World: Batch Data Loading

In [ ]:
r.flushdb()

# Load 10,000 records efficiently with pipeline batching
BATCH_SIZE = 1000
TOTAL_RECORDS = 10000

start = time.time()

for batch_start in range(0, TOTAL_RECORDS, BATCH_SIZE):
    with r.pipeline(transaction=False) as pipe:
        for i in range(batch_start, min(batch_start + BATCH_SIZE, TOTAL_RECORDS)):
            pipe.hset(f'record:{i}', mapping={
                'id': str(i),
                'value': f'data_{i}',
                'status': 'active'
            })
        pipe.execute()

elapsed = time.time() - start
print(f"Loaded {TOTAL_RECORDS} records in {elapsed:.2f}s")
print(f"Rate: {TOTAL_RECORDS/elapsed:.0f} records/sec")
print(f"Total keys: {r.dbsize()}")

---
## Pipelines vs Transactions

| Feature | Pipeline | Transaction |
|---|---|---|
| Purpose | Performance (batching) | Atomicity (safety) |
| Atomic? | No | Yes |
| Commands | `pipeline(transaction=False)` | `pipeline()` + `multi()` |
| Use case | Bulk inserts, batch reads | Money transfers, inventory |
| WATCH support | No | Yes |
| Speed benefit | Major | Minor (still one round trip) |

### When to Use What
- **Pipeline:** Loading lots of data, reading many keys, any bulk operation
- **Transaction:** When atomicity matters (money, inventory, counters)
- **WATCH + Transaction:** When you need check-and-set (read a value, decide, then write)

---
## Cleanup

In [ ]:
r.flushdb()
print("Cleaned up!")

---
## Key Takeaways

```
# Pipeline (batching for speed)
pipe = r.pipeline(transaction=False)
pipe.set('a', 1)  # queued
pipe.get('a')      # queued
results = pipe.execute()  # all sent at once

# Transaction (atomicity)
pipe = r.pipeline()  # transaction=True default
pipe.multi()
pipe.set('a', 1)
pipe.set('b', 2)
pipe.execute()  # atomic

# WATCH (optimistic locking)
pipe.watch('key')
val = pipe.get('key')  # read
pipe.multi()
pipe.set('key', new_val)  # write
pipe.execute()  # WatchError if key changed
```

---
## Exercises

1. **Pipeline Benchmark:** Compare pipelining vs individual commands for 10,000 GET operations. Measure and report the speedup.

2. **Safe Counter:** Implement a counter that increments only if the current value is below a maximum (e.g., max 100). Use WATCH to handle concurrent access.

3. **Seat Reservation:** Build a seat reservation system. Seats are represented as hash fields (`seat:A1`, `seat:A2`...). Use WATCH to safely reserve a seat only if it's not taken.

4. **Batch Import with Progress:** Load 50,000 records using pipelines. Print progress every 10,000 records with elapsed time and records/sec.

5. **Token Bucket Rate Limiter:** Implement a token bucket using WATCH. The bucket has max N tokens, refills at R tokens/sec. Each request consumes 1 token.

In [ ]:
# Your exercises here!


---
**Next up: [Notebook 09 — Persistence](./09_Persistence.ipynb)** — How Redis saves data to disk with RDB snapshots and AOF logs!